# Jersey Number Recognition via Keyframe Identification
### Implementation of Balaji, Bright et al. — *arXiv:2309.06285*

This notebook contains the full implementation:
1. **Configuration** — all hyper-parameters in one place
2. **KfId Module** — JNL → RoI → LHC → GHC pipeline (Section 3.1)
3. **Spatio-Temporal Network** — ResNet-18 + Bi-LSTM + multi-task heads (Section 3.2)
4. **SoccerNet Dataset** — loader that reads the official `gt.json` layout
5. **Training** — iteration-based loop matching Section 4.2 exactly
6. **Evaluation** — digit-wise accuracy on test split
7. **Inference** — single-tracklet prediction helper
8. **Smoke Test** — verify everything runs without a real dataset

## Imports

In [1]:
import os
import cv2
import math
import json
import random
import argparse
import numpy as np
from glob import glob
from pathlib import Path
from tqdm import tqdm
from sklearn.cluster import KMeans

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

## Configuration

All hyper-parameters from the paper are centralised here.  

In [2]:
class CFG:
    # image/sequence
    IMG_W           = 150
    IMG_H           = 120
    SEQ_LEN         = 40          # optimal per Table 6
    MIN_FRAME_GAP   = 2           # any 2 sampled frames >= d apart

    # training
    BATCH_SIZE      = 32
    LR              = 3e-3
    ITERATIONS      = 20_000
    LR_DECAY_STEP   = 2000        # reduce LR every 2000 iters …
    LR_DECAY_STOP   = 6000        # … only for the first 6000 iters
    DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

    # classification
    NUM_CLASSES_DIGIT = 11        # 0-9 digits + blank class (index 10)
    EPS               = 1e-7

    # KfId: RoI
    # top-left = (w/4, h/5),  bottom-right = (3w/4, h/2)
    ROI_TL_FRAC       = (0.25, 0.20)
    ROI_BR_FRAC       = (0.75, 0.50)
    ROI_THRESHOLD     = 0.3       # minimum I* score to keep a detection

    # KfId: LHC 
    LHC_CORR_THRESHOLD = 0.7      # hue-histogram correlation threshold
    LHC_PROXIMITY_PX   = 30       # max pixel distance for merging digits

    # KfId: GHC 
    GHC_N_CLUSTERS     = 3        # KMeans clusters over hue histograms

    # model dimensions
    SPATIAL_FEAT_DIM   = 512      # ResNet-18 penultimate layer
    LSTM_HIDDEN        = 256      # total (128 per direction × 2)
    LSTM_LAYERS        = 1

    # data
    DATA_ROOT = "SoccerNet/train/train"


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
print(f"Device: {CFG.DEVICE}")

Device: cpu


## Keyframe Identification Module (Section 3.1)

The KfId pipeline filters a raw tracklet down to only those frames where
the target player's jersey number is actually visible:

```
For each frame:
    JNL  →  detect all digits
    RoI  →  discard detections outside the jersey region
    LHC  →  merge nearby same-colour digit crops into one holistic box

Across all frames:
    GHC  →  cluster hue histograms; keep dominant cluster (= target player)
```

### Jersey Number Localizer (JNL)

In [3]:
class JerseyNumberLocalizer:
    def __init__(self, weights_path: str | None = None):
        self.model = None
        if weights_path and os.path.exists(weights_path):
            try:
                self.model = torch.hub.load(
                    "ultralytics/yolov5", "custom",
                    path=weights_path, force_reload=False,
                )
                self.model.eval()
                print(f"[JNL] Loaded detector from {weights_path}")
            except Exception as e:
                print(f"[JNL] Could not load detector: {e}")

    def detect(self, frame_bgr: np.ndarray) -> list[list[int]]:
        if self.model is None:
            h, w = frame_bgr.shape[:2]
            return [[0, 0, w, h]]          # stub fallback

        results = self.model(frame_bgr)
        return [[int(v) for v in xyxy]
                for *xyxy, conf, cls in results.xyxy[0].cpu().numpy()]

### RoI-based Filtering (Section 3.1.2)

In [4]:
class RoIFilter:
    def __init__(self, img_w: int, img_h: int):
        self.roi = (
            int(CFG.ROI_TL_FRAC[0] * img_w),   # x1
            int(CFG.ROI_TL_FRAC[1] * img_h),   # y1
            int(CFG.ROI_BR_FRAC[0] * img_w),   # x2
            int(CFG.ROI_BR_FRAC[1] * img_h),   # y2
        )

    @staticmethod
    def _area(box) -> int:
        return max(0, box[2] - box[0]) * max(0, box[3] - box[1])

    def _i_star(self, r1, r2) -> float:
        inter = (
            max(r1[0], r2[0]), max(r1[1], r2[1]),
            min(r1[2], r2[2]), min(r1[3], r2[3]),
        )
        return self._area(inter) / (min(self._area(r1), self._area(r2)) + CFG.EPS)

    def filter(self, detections: list[list[int]]) -> list[list[int]]:
        return [d for d in detections
                if self._i_star(self.roi, tuple(d)) >= CFG.ROI_THRESHOLD]

### Local Histogram Correlation (LHC — Section 3.1.3)

In [5]:
class LocalHistogramCorrelation:
    @staticmethod
    def _hue_hist(crop_bgr: np.ndarray, bins: int = 36) -> np.ndarray:
        hsv  = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2HSV)
        hist, _ = np.histogram(hsv[:, :, 0], bins=bins, range=(0, 180))
        hist = hist.astype(np.float32)
        return hist / (hist.sum() + CFG.EPS)

    @staticmethod
    def _center(box) -> tuple:
        return ((box[0] + box[2]) / 2, (box[1] + box[3]) / 2)

    @staticmethod
    def _merge(b1, b2) -> list:
        return [min(b1[0], b2[0]), min(b1[1], b2[1]),
                max(b1[2], b2[2]), max(b1[3], b2[3])]

    def process(self, frame_bgr: np.ndarray,
                detections: list[list[int]]) -> list[dict]:
        if not detections:
            return []

        h, w = frame_bgr.shape[:2]
        crops = []
        for d in detections:
            x1, y1 = max(0, d[0]), max(0, d[1])
            x2, y2 = min(w, d[2]), min(h, d[3])
            crop = frame_bgr[y1:y2, x1:x2]
            crops.append(crop if crop.size > 0
                         else np.zeros((10, 10, 3), dtype=np.uint8))

        hists = [self._hue_hist(c) for c in crops]
        used  = [False] * len(detections)
        merged = []

        for i in range(len(detections)):
            if used[i]:
                continue
            box_i, hist_i = list(detections[i]), hists[i].copy()
            for j in range(i + 1, len(detections)):
                if used[j]:
                    continue
                dist = math.hypot(*(a - b for a, b in
                                    zip(self._center(box_i),
                                        self._center(detections[j]))))
                corr = float(np.corrcoef(hist_i, hists[j])[0, 1])
                if dist < CFG.LHC_PROXIMITY_PX and corr > CFG.LHC_CORR_THRESHOLD:
                    box_i  = self._merge(box_i, detections[j])
                    hist_i = (hist_i + hists[j]) / 2
                    used[j] = True
            used[i] = True
            merged.append({"box": box_i, "hist": hist_i})

        return merged

### Global Histogram Correlation (GHC — Section 3.1.3)

In [6]:
class GlobalHistogramCorrelation:
    def process(self, tracklet_dets: list[list[dict]]) -> list[list[dict]]:
        flat_hists, flat_idx = [], []
        for fi, frame_dets in enumerate(tracklet_dets):
            for di, det in enumerate(frame_dets):
                flat_hists.append(det["hist"])
                flat_idx.append((fi, di))

        if len(flat_hists) < 2:
            return tracklet_dets

        n_clusters = min(CFG.GHC_N_CLUSTERS, len(flat_hists))
        X      = np.stack(flat_hists)
        labels = KMeans(n_clusters=n_clusters, n_init=10,
                        random_state=42).fit_predict(X)

        dominant = int(np.argmax(np.bincount(labels, minlength=n_clusters)))
        keep = {flat_idx[k] for k, lbl in enumerate(labels) if lbl == dominant}

        return [
            [det for di, det in enumerate(frame_dets) if (fi, di) in keep]
            for fi, frame_dets in enumerate(tracklet_dets)
        ]

### Full KfId Pipeline

In [7]:
class KeyframeIdentificationModule:
    def __init__(self, img_w: int = CFG.IMG_W, img_h: int = CFG.IMG_H,
                 detector_weights: str | None = None):
        self.jnl = JerseyNumberLocalizer(detector_weights)
        self.roi = RoIFilter(img_w, img_h)
        self.lhc = LocalHistogramCorrelation()
        self.ghc = GlobalHistogramCorrelation()

    def process_tracklet(
        self, frames: list[np.ndarray]
    ) -> tuple[list[int], list[dict]]:
        # Eq. 1 + 2: per-frame JNL → RoI → LHC
        local_filtered = [
            self.lhc.process(f, self.roi.filter(self.jnl.detect(f)))
            for f in frames
        ]

        # Eq. 3: cross-frame GHC
        global_filtered = self.ghc.process(local_filtered)

        # Collect frames with at least one valid detection
        keyframe_indices, keyframe_boxes = [], []
        for fi, dets in enumerate(global_filtered):
            if dets:
                keyframe_indices.append(fi)
                keyframe_boxes.append(dets[0])

        return keyframe_indices, keyframe_boxes

## Spatio-Temporal Network (Section 3.2)

```
Input (B, T, 3, H, W)
    ↓  ResNet-18 applied to each frame independently
512-d spatial features  (B, T, 512)
    ↓  Bi-LSTM
256-d temporal features (B, 256)  — mean-pooled over time
    ↓  Two linear heads (digit-wise)
logits_d1 (B, 11),  logits_d2 (B, 11)
```

### Spatial Encoder — ResNet-18

In [8]:
class SpatialEncoder(nn.Module):
    def __init__(self, pretrained: bool = True):
        super().__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet18(weights=weights)
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        self.out_dim  = 512

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 3, H, W)  →  out: (B, 512)
        return self.features(x).flatten(1)

### Full Spatio-Temporal Network

In [9]:
class SpatioTemporalNetwork(nn.Module):
    def __init__(self, pretrained_resnet: bool = True):
        super().__init__()
        self.spatial_encoder = SpatialEncoder(pretrained=pretrained_resnet)

        # Bi-LSTM: hidden=128 per direction → 256-d output total
        lstm_hidden = CFG.LSTM_HIDDEN // 2
        self.bilstm = nn.LSTM(
            input_size=CFG.SPATIAL_FEAT_DIM,
            hidden_size=lstm_hidden,
            num_layers=CFG.LSTM_LAYERS,
            batch_first=True,
            bidirectional=True,
        )
        temporal_dim = lstm_hidden * 2   # 256

        # independent digit heads (section 3.2 + Table 5 ablation)
        self.head_d1 = nn.Linear(temporal_dim, CFG.NUM_CLASSES_DIGIT)
        self.head_d2 = nn.Linear(temporal_dim, CFG.NUM_CLASSES_DIGIT)

    def forward(
        self, x: torch.Tensor
        ) -> tuple[torch.Tensor, torch.Tensor]:
        
        B, T, C, H, W = x.shape
        spatial = self.spatial_encoder(x.view(B * T, C, H, W))   # (B*T, 512)
        spatial = spatial.view(B, T, -1)                          # (B, T, 512)

        lstm_out, _ = self.bilstm(spatial)                        # (B, T, 256)
        temporal    = lstm_out.mean(dim=1)                        # (B, 256)

        return self.head_d1(temporal), self.head_d2(temporal)

### Multi-Task Loss (Equations 5–7)

In [10]:
class MultiTaskLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()

    def forward(self, logits_d1, logits_d2, targets_d1, targets_d2):
        return 0.5 * self.ce(logits_d1, targets_d1) +                0.5 * self.ce(logits_d2, targets_d2)

## 4 · SoccerNet Dataset

```
SoccerNet/
    train/
        train/
            gt.json          {"player_id": jersey_number, ...}
            images/
                <player_id>/
                    0_1.jpg
                    0_2.jpg  ...
    test/
        gt.json
        images/ ...
    challenge/          
        images/ ...
```

### Label helper

In [11]:
def jersey_label_to_digits(label: int) -> tuple[int, int]:
    s = str(label)
    if len(s) == 1:
        return int(s[0]), 10
    elif len(s) == 2:
        return int(s[0]), int(s[1])
    raise ValueError(f"Unexpected jersey number: {label}")


# labels in gt.json that mean "jersey not visible"
_UNKNOWN_LABELS = {-1, 0, None}

### SoccerNetDataset

In [12]:
class SoccerNetDataset(Dataset):
    # grayscale only
    TRANSFORM = T.Compose([
        T.ToPILImage(),
        T.Resize((CFG.IMG_H, CFG.IMG_W)),
        T.Grayscale(num_output_channels=3),   # 3-ch grayscale for ResNet
        T.ToTensor(),
    ])

    def __init__(self, split_dir: str, seq_len: int = CFG.SEQ_LEN,
                 min_gap: int = CFG.MIN_FRAME_GAP, kfid=None):
        self.seq_len = seq_len
        self.min_gap = min_gap
        self.kfid    = kfid
        self.samples = self._build_index(Path(split_dir))
        print(f"Loaded {len(self.samples)} tracklets from '{split_dir}'")

    def _build_index(self, split_dir: Path) -> list[dict]:
        split_name = split_dir.name
        candidates = [
            split_dir / f"{split_name}_gt.json",
            split_dir / "gt.json",
        ]
        gt_path = next((p for p in candidates if p.exists()), None)
        if gt_path is None:
            raise FileNotFoundError(
                f"Could not find a GT JSON in {split_dir}.\n"
                f"Tried: {[str(p) for p in candidates]}\n"
                "Challenge split has no labels; use it for inference only."
            )

        with open(gt_path) as f:
            gt = json.load(f)

        images_root = split_dir / "images"
        samples = []
        for player_id, jersey_num in gt.items():
            tracklet_dir = images_root / str(player_id)
            if not tracklet_dir.exists():
                continue
            frame_paths = sorted(
                glob(str(tracklet_dir / "*.jpg")) +
                glob(str(tracklet_dir / "*.png"))
            )
            if not frame_paths:
                continue
            if jersey_num in _UNKNOWN_LABELS:
                d1, d2 = 10, 10
            else:
                try:
                    d1, d2 = jersey_label_to_digits(int(jersey_num))
                except ValueError:
                    d1, d2 = 10, 10
            samples.append({"player_id": player_id,
                             "frame_paths": frame_paths,
                             "d1": d1, "d2": d2})
        return samples

    def _load_frames(self, paths: list[str]) -> list[np.ndarray]:
        frames = []
        for p in paths:
            img = cv2.imread(p)
            if img is None:
                img = np.zeros((CFG.IMG_H, CFG.IMG_W, 3), dtype=np.uint8)
            frames.append(img)
        return frames

    def _sample_with_gap(self, pool: list[int]) -> list[int]:
        extended = pool[:]
        while len(extended) < self.seq_len:
            extended += pool

        random.shuffle(extended)
        selected = []
        for idx in sorted(extended):
            if not selected or (idx - selected[-1]) >= self.min_gap:
                selected.append(idx)
            if len(selected) == self.seq_len:
                break

        while len(selected) < self.seq_len:
            selected.append(selected[-1])
        return selected[: self.seq_len]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict:
        sample     = self.samples[idx]
        all_frames = self._load_frames(sample["frame_paths"])

        # KfId filtering (skip if kfid is None)
        if self.kfid is not None:
            kf_indices, _ = self.kfid.process_tracklet(all_frames)
            pool = kf_indices if kf_indices else list(range(len(all_frames)))
        else:
            pool = list(range(len(all_frames)))

        chosen     = self._sample_with_gap(pool)
        frames     = [all_frames[i] for i in chosen]
        tensors    = [self.TRANSFORM(f) for f in frames]
        seq_tensor = torch.stack(tensors, dim=0)   # (T, 3, H, W)

        return {
            "frames":    seq_tensor,
            "d1":        torch.tensor(sample["d1"], dtype=torch.long),
            "d2":        torch.tensor(sample["d2"], dtype=torch.long),
            "player_id": sample["player_id"],
        }

### Dataloader

In [13]:
train_ds = SoccerNetDataset( 
    split_dir= CFG.DATA_ROOT, 
    kfid=None, 
)
test_ds = SoccerNetDataset(
    split_dir="SoccerNet/test/test", 
    kfid=None,
)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=(CFG.DEVICE == "cuda"),
    drop_last=True,    
)
test_loader = DataLoader(
    test_ds,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=(CFG.DEVICE == "cuda"),
)

print(f"Train batches : {len(train_loader)}")
print(f"Test  batches : {len(test_loader)}")

Loaded 1427 tracklets from 'SoccerNet/train/train'
Loaded 1211 tracklets from 'SoccerNet/test/test'
Train batches : 44
Test  batches : 38


## Training (Section 4.2)

- Optimiser: Adam, lr = 3e-3
- LR schedule: ×0.1 at iterations 2 000, 4 000, 6 000 (then frozen)
- Total iterations: 20 000
- Batch size: 32
- Checkpoint saved every 5 000 iterations

In [14]:
def run_training(
    model,
    train_loader,
    n_iterations:     int   = CFG.ITERATIONS,
    lr:               float = CFG.LR,
    device:           str   = CFG.DEVICE,
    checkpoint_dir:   str   = "checkpoints",
    checkpoint_every: int   = 5_000,
) -> list[float]:

    os.makedirs(checkpoint_dir, exist_ok=True)
    model.to(device)

    criterion = MultiTaskLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    milestones = list(range(CFG.LR_DECAY_STEP,
                            CFG.LR_DECAY_STOP + 1,
                            CFG.LR_DECAY_STEP))   # [2000, 4000, 6000]
    scheduler = optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=milestones, gamma=0.1
    )

    model.train()
    losses    = []
    data_iter = iter(train_loader)
    pbar      = tqdm(range(n_iterations), desc="Training")

    for iteration in pbar:
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            batch = next(data_iter)

        frames = batch["frames"].to(device)
        d1, d2 = batch["d1"].to(device), batch["d2"].to(device)

        optimizer.zero_grad()
        logits_d1, logits_d2 = model(frames)
        loss = criterion(logits_d1, logits_d2, d1, d2)
        loss.backward()
        optimizer.step()

        if iteration < CFG.LR_DECAY_STOP:
            scheduler.step()

        losses.append(loss.item())
        pbar.set_postfix(loss=f"{loss.item():.4f}",
                         lr=f"{optimizer.param_groups[0]['lr']:.1e}")

        if (iteration + 1) % checkpoint_every == 0:
            ckpt = os.path.join(checkpoint_dir,
                                f"model_iter{iteration + 1}.pt")
            torch.save({"iteration": iteration + 1,
                        "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "loss": loss.item()}, ckpt)
            print(f"\n  ✓ Checkpoint → {ckpt}")

    final = os.path.join(checkpoint_dir, "model_final.pt")
    torch.save(model.state_dict(), final)
    print(f"\n  ✓ Final model → {final}")
    return losses

### Run Training Model

In [ ]:
model = SpatioTemporalNetwork(pretrained_resnet=True)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

losses = run_training(model, train_loader)

Parameters: 11,839,574


## Evaluation
Target performance (Table 4): **68.53% test accuracy**.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device: str = CFG.DEVICE) -> dict:
    model.eval()
    model.to(device)
    correct = total = 0

    for batch in tqdm(loader, desc="Evaluating"):
        frames = batch["frames"].to(device)
        gt_d1, gt_d2 = batch["d1"].to(device), batch["d2"].to(device)

        pred_d1 = model(frames)[0].argmax(dim=-1)
        pred_d2 = model(frames)[1].argmax(dim=-1)

        correct += ((pred_d1 == gt_d1) & (pred_d2 == gt_d2)).sum().item()
        total   += frames.size(0)

    acc = 100.0 * correct / max(total, 1)
    print(f"Accuracy: {correct}/{total} = {acc:.2f}%")
    return {"accuracy": acc, "correct": correct, "total": total}

results = evaluate(model, test_loader)

## 7 · Single-Tracklet Inference

Use `predict_tracklet` to get a jersey number for one player given their
list of raw BGR frames (e.g. loaded directly from video).

In [ ]:
def predict_tracklet(
    model,
    frames: list[np.ndarray],
    kfid: KeyframeIdentificationModule,
    seq_len: int = CFG.SEQ_LEN,
    device:  str = CFG.DEVICE,
) -> int:
    model.eval()
    model.to(device)

    kf_indices, _ = kfid.process_tracklet(frames)
    pool = kf_indices if kf_indices else list(range(len(frames)))

    # pad / sample to fixed length
    extended = pool[:]
    while len(extended) < seq_len:
        extended += pool
    selected = sorted(random.sample(extended, seq_len))

    tensors = [SoccerNetDataset.TRANSFORM(frames[i]) for i in selected]
    seq     = torch.stack(tensors).unsqueeze(0).to(device)  # (1, T, 3, H, W)

    with torch.no_grad():
        logits_d1, logits_d2 = model(seq)
    d1 = logits_d1.argmax(dim=-1).item()
    d2 = logits_d2.argmax(dim=-1).item()

    return d1 if d2 == 10 else d1 * 10 + d2


# ── Example usage ────────────────────────────────────────────────────
# frames = [cv2.imread(p) for p in sorted(glob("player_tracklet/*.jpg"))]
# kfid   = KeyframeIdentificationModule(detector_weights="yolov5_jersey.pt")
# number = predict_tracklet(model, frames, kfid)
# print(f"Predicted jersey: #{number}")

## 8 · Smoke Test

Verifies that every component runs end-to-end without a real dataset.

In [ ]:
print("=" * 55)
print("Smoke Test — no real data needed")
print("=" * 55)

# 1. KfId on random frames
print("\n[1] KfId module (stub detector — no weights)")
kfid   = KeyframeIdentificationModule()
dummy  = [np.random.randint(0, 255, (CFG.IMG_H, CFG.IMG_W, 3),
          dtype=np.uint8) for _ in range(20)]
kf_idx, _ = kfid.process_tracklet(dummy)
print(f"    20 input frames  →  {len(kf_idx)} keyframes")

# 2. Model forward pass
print("\n[2] SpatioTemporalNetwork forward pass")
_model = SpatioTemporalNetwork(pretrained_resnet=False)
x = torch.randn(2, CFG.SEQ_LEN, 3, CFG.IMG_H, CFG.IMG_W)
l1, l2 = _model(x)
print(f"    Input : {tuple(x.shape)}")
print(f"    d1 logits : {tuple(l1.shape)}")
print(f"    d2 logits : {tuple(l2.shape)}")

# 3. Loss
print("\n[3] MultiTaskLoss")
criterion = MultiTaskLoss()
loss = criterion(l1, l2,
                 torch.randint(0, 11, (2,)),
                 torch.randint(0, 11, (2,)))
print(f"    Loss value: {loss.item():.4f}")

# 4. Label conversion
print("\n[4] Label conversion")
for num in [7, 34, 99]:
    d1, d2 = jersey_label_to_digits(num)
    print(f"    #{num:2d}  →  d1={d1}, d2={d2}")

print("\n✓ All components initialised successfully.")